In [4]:
import pandas as pd
import os
from pathlib import Path

cards = pd.read_csv(Path.cwd().parent.parent / "archive" / "cards_data.csv")
cards.head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [5]:
transactions = pd.read_csv(Path.cwd().parent.parent / "archive" / "transactions_data.csv")
transactions.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [6]:
users = pd.read_csv(Path.cwd().parent.parent / "archive" / "users_data.csv")
users.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [9]:
import json

path = Path.cwd().parent.parent / "archive" / "train_fraud_labels.json"

with open(path, "r") as f:
    raw = json.load(f)
train_fraud_labels = pd.DataFrame.from_dict(raw, orient="index")
train_fraud_labels.T.head()

,target
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No


In [10]:
train_fraud_labels = train_fraud_labels.T
train_fraud_labels.index.name = "transaction_id"
train_fraud_labels.head()

,target
transaction_id,
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No


In [14]:
# Check dtypes
print("Labels index dtype:", train_fraud_labels.index.dtype)
print("Transactions id dtype:", transactions["id"].dtype)

# Check overlap
labels_ids = set(train_fraud_labels.index.astype(str))
transaction_ids = set(transactions["id"].astype(str))

print("Total label IDs:", len(labels_ids))
print("Total transaction IDs:", len(transaction_ids))
print("Overlapping IDs:", len(labels_ids & transaction_ids))
print("In labels but not transactions:", len(labels_ids - transaction_ids))
print("In transactions but not labels:", len(transaction_ids - labels_ids))

Labels index dtype: object
Transactions id dtype: int64
Total label IDs: 8914963
Total transaction IDs: 13305915
Overlapping IDs: 8914963
In labels but not transactions: 0
In transactions but not labels: 4390952


In [22]:
transactions["id"] = transactions["id"].astype(str)
train_fraud_labels.index = train_fraud_labels.index.astype(str)

df = transactions.merge(
    train_fraud_labels, left_on="id", right_index=True, how="left"
)

In [24]:
print(len(df))

13305915


In [25]:
df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,transaction_id,transaction_id,transaction_id,transaction_id,transaction_id,target
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN,7475327.0,7475327.0,7475327.0,7475327.0,19088647.0,0.0
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN,7475328.0,7475328.0,7475328.0,7475328.0,12988914.0,0.0
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN,7475329.0,7475329.0,7475329.0,7475329.0,18475617.0,0.0
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN,7475331.0,7475331.0,7475331.0,7475331.0,7855340.0,0.0
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN,7475332.0,7475332.0,7475332.0,7475332.0,21750405.0,0.0


In [8]:
path = Path.cwd().parent.parent / "archive" / "mcc_codes.json"

with open(path, "r") as f:
    raw = json.load(f)
mcc_codes = pd.DataFrame.from_dict(raw, orient="index")
mcc_codes.head()

,0
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees
